# Тренування TimeSformer на Google Colab T4

Цей ноутбук виконує повний цикл тренування і оцінки моделі для проєкту
**Tactical Gesture Recognition System** на безкоштовному GPU T4 у Colab.

## Що відбувається

1. Перевірка наявності GPU (T4).
2. Монтування Google Drive (де лежить датасет і код).
3. Встановлення залежностей з `requirements-colab.txt` (CUDA-білди torch).
4. Перевірка цілісності датасету (`tools.verify_dataset`).
5. Тренування моделі (`src.train`) — ~30–60 хв на T4.
6. Оцінка на test-підмножині (`src.evaluate`).
7. Замір GPU latency (`tools.profile_inference`).
8. Синхронізація результатів назад на Google Drive.

## Передумови

Перед запуском ноутбука переконайтесь, що:

- **Датасет** залитий у `MyDrive/tactical_gestures/data/<class>/clip_NNN.mp4`.
- **Код проєкту** — також у `MyDrive/tactical_gestures/` (папки `src/`,
  `tools/`, файл `config.py`, `requirements-colab.txt`).
  Якщо коду немає на Drive — ноутбук запропонує завантажити zip-архів.

⚠️ **Увага:** безкоштовна сесія Colab триває до 12 годин і очищує
`/content/` при завершенні — тому останній крок копіює артефакти назад
на Drive.

## 1. Перевірка GPU

Перед усім переконуємось, що Colab виділив нам GPU. Якщо команда
`nvidia-smi` повертає помилку, змініть тип середовища:
**Runtime → Change runtime type → Hardware accelerator: GPU**, тоді
перезапустіть ноутбук.

In [ ]:
!nvidia-smi

## 2. Підключення Google Drive

Після виконання комірки з'явиться діалог авторизації — підтвердьте
доступ до Drive. Drive змонтується в `/content/drive/`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Розташування проєкту

Якщо проєкт уже залитий на Drive у `MyDrive/tactical_gestures/`,
використовуємо його. Інакше пропонуємо завантажити zip-архів проєкту
через діалоговий файловий вибір — він розпакується в `/content/`.

In [ ]:
import os
import sys
import zipfile
from pathlib import Path

drive_root = Path('/content/drive/MyDrive/tactical_gestures')
if (drive_root / 'src').is_dir() and (drive_root / 'config.py').is_file():
    project_root = drive_root
    print(f'✅ Проєкт знайдено на Drive: {project_root}')
else:
    from google.colab import files

    print('⚠️ Проєкт не знайдено в /MyDrive/tactical_gestures/.')
    print('Завантажте zip-архів проєкту (з src/, tools/, config.py):')
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    extract_dir = Path('/content/tactical_gestures')
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(extract_dir)
    entries = [e for e in extract_dir.iterdir() if e.is_dir()]
    if len(entries) == 1 and (entries[0] / 'src').is_dir():
        project_root = entries[0]
    else:
        project_root = extract_dir
    print(f'✅ Розпаковано: {project_root}')

os.chdir(project_root)
sys.path.insert(0, str(project_root))
print(f'CWD: {Path.cwd()}')

## 4. Встановлення залежностей

`requirements-colab.txt` дотягує CUDA-білди `torch` і `torchvision`,
які підходять для T4 GPU.

In [ ]:
!pip install --quiet -r requirements-colab.txt

## 5. Smoke-import модулів

Перевіряємо, що пакет `src` і `config` доступні з PYTHONPATH і що
CUDA дійсно бачить наш T4.

In [ ]:
import torch

import config
from src import dataset, model, train  # noqa: F401 — smoke import

print(f'Жестів у config: {len(config.GESTURES)}')
print(f'CUDA доступна: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 6. Перевірка датасету

Перш ніж починати тренування, переконуємось, що всі кліпи читаються,
мають правильну тривалість і FPS, і немає сильного дисбалансу класів.
Якщо `verify_dataset` повертає ⚠️ — обговоріть проблеми перед тренуванням.

In [ ]:
!python -m tools.verify_dataset

## 7. Тренування

Параметри:

- `--epochs 20` — максимум 20 епох. Рання зупинка спрацює, якщо
  `val_acc` не покращується 5 епох поспіль.
- `--batch-size 8` — комфортний розмір для T4 (~15 GB VRAM). Якщо
  отримаєте CUDA OOM — знизьте до 4 або 2.
- `--num-workers 2` — паралельне завантаження кліпів на Colab/Linux.
- `--device cuda` — явно вказуємо GPU.

Очікувана тривалість: ~30–60 хв на T4 з датасетом ~100 кліпів.

Артефакти зберігаються в `checkpoints/`:

- `best_model.pt` — найкращий чекпойнт за `val_acc`;
- `last_model.pt` — стан після останньої епохи;
- `training_history.json` — метрики по епохах для побудови кривих.

In [ ]:
!python -m src.train --epochs 20 --batch-size 8 --num-workers 2 --device cuda

## 8. Оцінка на test-підмножині

`src.evaluate` створює у `evaluation_results/`:

- `metrics_test.json` — повний звіт (top-1, top-2, macro P/R/F1,
  per-class, confusion matrix).
- `metrics_test.csv` — wide-format таблиця для вставки в §4.4.
- 4 PNG-графіки: криві навчання, матриця помилок (counts + normalized),
  per-class F1.

In [ ]:
!python -m src.evaluate --device cuda --num-workers 2

## 9. Перегляд графіків

Виводимо результати прямо в ноутбук — корисно для швидкої візуальної
перевірки перед тим, як забирати файли на ноутбук Анастасії.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

plots = [
    'training_curves.png',
    'CM_counts.png',
    'CM_normalized.png',
    'per_class_f1.png',
]
for name in plots:
    path = Path('evaluation_results') / name
    if path.is_file():
        print(f'--- {name} ---')
        display(Image(str(path)))
    else:
        print(f'⚠️ Не знайдено: {path}')

## 10. Замір GPU latency

Це число для §4.4.6 тези — порівняння CPU vs GPU. На T4 одне
forward-проходження TimeSformer-Base триває приблизно 80–150 мс
(майже у 30× швидше за CPU).

In [ ]:
!python -m tools.profile_inference --device cuda --num-runs 50

## 11. Синхронізація на Google Drive

Колаб після завершення сесії очищує `/content/`. Копіюємо чекпойнти,
історію та результати назад на Drive, щоб вони залишилися доступні
з ноутбука Анастасії.

In [ ]:
!mkdir -p /content/drive/MyDrive/tactical_gestures/
!cp -r checkpoints/ /content/drive/MyDrive/tactical_gestures/
!cp -r evaluation_results/ /content/drive/MyDrive/tactical_gestures/
print('✅ Результати збережено на Google Drive: /MyDrive/tactical_gestures/')
print('   - checkpoints/best_model.pt, last_model.pt, training_history.json')
print('   - evaluation_results/metrics_test.json + .csv + 4 PNG + latency_cuda.json')

## Що далі

Після виконання ноутбука у вас на Drive є:

1. **Чекпойнти** (`checkpoints/`):
   - `best_model.pt` — використовуйте у локальному запуску
     (`python -m src.inference --checkpoint <path> --webcam 0 --audio`).
   - `training_history.json` — метрики по епохах для §4.4.4 тези.

2. **Результати оцінки** (`evaluation_results/`):
   - `metrics_test.json` — повний звіт у JSON.
   - `metrics_test.csv` — таблиця для вставки в §4.4.
   - `training_curves.png`, `CM_counts.png`, `CM_normalized.png`,
     `per_class_f1.png` — графіки для тези.
   - `latency_cuda.json` — GPU-latency для §4.4.6.

Завантажте папку `tactical_gestures/` з Drive назад на свій ноутбук
(через Drive File Stream або вручну) — і використовуйте артефакти для
написання Розділу 4.